In [1]:
import pandas as pd
import re

df = pd.read_csv("/content/nlp_customer_messages - nlp_customer_messages.csv")

print(df.head())

  message_id                                            message
0     MSG001  Good morning, My phone is not working :(  Than...
1     MSG002  Hi, Thanks for your quick response! I need hel...
2     MSG003  Hi support, I LOVE this product 😍😍 Can someone...
3     MSG004  Hey there, The new update is AMAZING!! Please ...
4     MSG005  Dear team, The price is 250 EGP, too expensive...


In [2]:
print("Shape:", df.shape)

print("Columns:", df.columns.tolist())

Shape: (80, 2)
Columns: ['message_id', 'message']


In [3]:
print("Missing values:")
print(df.isnull().sum())

print("Duplicated message_id:")
print(df["message_id"].duplicated().sum())

Missing values:
message_id    0
message       0
dtype: int64
Duplicated message_id:
0


In [4]:
df["char_count"] = df["message"].str.len()

print(df[["message_id", "message", "char_count"]].head())

  message_id                                            message  char_count
0     MSG001  Good morning, My phone is not working :(  Than...          55
1     MSG002  Hi, Thanks for your quick response! I need hel...          55
2     MSG003  Hi support, I LOVE this product 😍😍 Can someone...          84
3     MSG004  Hey there, The new update is AMAZING!! Please ...          96
4     MSG005  Dear team, The price is 250 EGP, too expensive...          81


In [5]:
longest_index = df["char_count"].idxmax()
shortest_index = df["char_count"].idxmin()

print("Longest message:")
print(df.loc[longest_index, ["message_id", "message", "char_count"]])

print("\nShortest message:")
print(df.loc[shortest_index, ["message_id", "message", "char_count"]])

Longest message:
message_id                                               MSG036
message       Hey there, I need help with payment for order ...
char_count                                                  209
Name: 35, dtype: object

Shortest message:
message_id                                            MSG029
message       Dear team, Thanks! Great service :) Thank you.
char_count                                                46
Name: 28, dtype: object


In [7]:
print("Number of messages:", len(df))
print("Missing messages:", df["message"].isnull().sum())
print("Duplicated IDs:", df["message_id"].duplicated().sum())
print("Average message length:", df["char_count"].mean())

Number of messages: 80
Missing messages: 0
Duplicated IDs: 0
Average message length: 96.3625


Summary
The dataset contains 80 fictional customer messages. I checked the dataset shape, column names, missing values, and duplicated message IDs. I also created a char_count column to measure message length without modifying the original columns.

In [8]:
email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

url_pattern = r"https?://[^\s\]\)<>]+"

phone_pattern = r"\b010\d{8}\b"

In [9]:
def extract_emails(text):
    return re.findall(email_pattern, text)


def extract_urls(text):
    return re.findall(url_pattern, text)


def extract_phones(text):
    return re.findall(phone_pattern, text)


def extract_numbers(text):
    return re.findall(r"\d+", text)


def check_urgent(text):
    return bool(re.search(r"\burgent\b", text, re.IGNORECASE))

In [10]:
df["emails"] = df["message"].apply(extract_emails)
df["urls"] = df["message"].apply(extract_urls)
df["phones"] = df["message"].apply(extract_phones)
df["numbers"] = df["message"].apply(extract_numbers)
df["has_urgent"] = df["message"].apply(check_urgent)

In [11]:
mask = (
    df["emails"].apply(bool)
    | df["urls"].apply(bool)
    | df["phones"].apply(bool)
)

print(df.loc[
    mask,
    ["message_id", "message", "emails", "urls", "phones", "numbers", "has_urgent"]
].head(10))

   message_id                                            message  \
2      MSG003  Hi support, I LOVE this product 😍😍 Can someone...   
3      MSG004  Hey there, The new update is AMAZING!! Please ...   
4      MSG005  Dear team, The price is 250 EGP, too expensive...   
5      MSG006  Dear support, The package for order 10102 is d...   
7      MSG008  Hello team, Great quality, but delivery took 7...   
8      MSG009  Good morning, Thanks! Great service :) Thanks ...   
9      MSG010  Hi, Can you change my delivery address? I need...   
11     MSG012  Hey there, I cannot log in to my account. Plea...   
14     MSG015  Hello, The delivery was FAST!!! What should I ...   
15     MSG016  Hello team, I need help with payment for order...   

                                               emails  \
2                             [customer3@example.com]   
3                                                  []   
4                                                  []   
5                      

In [12]:
print("Messages with email:", df["emails"].apply(bool).sum())

print("Messages with phone:", df["phones"].apply(bool).sum())

print("Messages with URL:", df["urls"].apply(bool).sum())

print("Messages with urgent:", df["has_urgent"].sum())

Messages with email: 26
Messages with phone: 16
Messages with URL: 20
Messages with urgent: 11


In [13]:
def clean_text(text):
    text = re.sub(email_pattern, "[EMAIL]", text)
    text = re.sub(url_pattern, "[URL]", text)
    text = re.sub(phone_pattern, "[PHONE]", text)
    text = re.sub(r"\d+", "[NUMBER]", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text

In [14]:
df["clean_message"] = df["message"].apply(clean_text)

print(df[["message", "clean_message"]].head(8).to_string(index=False))

                                                                                                                           message                                                                                                   clean_message
                                                                           Good morning, My phone is not working :(  Thanks again!                                                          Good morning, My phone is not working :( Thanks again!
                                                                           Hi, Thanks for your quick response! I need help ASAP!!!                                                         Hi, Thanks for your quick response! I need help ASAP!!!
                                              Hi support, I LOVE this product 😍😍 Can someone help me? Email: customer3@example.com                                          Hi support, I LOVE this product 😍😍 Can someone help me? Email: [EMAIL]
                            

In [15]:
test_messages = [
    "Please email me at test@example.com",
    "Call me at 01012345678",
    "Visit https://example.com/help",
    "Hello, I need help with my order!"
]

In [16]:
for text in test_messages:
    print("Original:", text)
    print("Emails:", extract_emails(text))
    print("URLs:", extract_urls(text))
    print("Phones:", extract_phones(text))
    print("Numbers:", extract_numbers(text))
    print("Urgent:", check_urgent(text))
    print("Cleaned:", clean_text(text))
    print("-" * 50)

Original: Please email me at test@example.com
Emails: ['test@example.com']
URLs: []
Phones: []
Numbers: []
Urgent: False
Cleaned: Please email me at [EMAIL]
--------------------------------------------------
Original: Call me at 01012345678
Emails: []
URLs: []
Phones: ['01012345678']
Numbers: ['01012345678']
Urgent: False
Cleaned: Call me at [PHONE]
--------------------------------------------------
Original: Visit https://example.com/help
Emails: []
URLs: ['https://example.com/help']
Phones: []
Numbers: []
Urgent: False
Cleaned: Visit [URL]
--------------------------------------------------
Original: Hello, I need help with my order!
Emails: []
URLs: []
Phones: []
Numbers: []
Urgent: False
Cleaned: Hello, I need help with my order!
--------------------------------------------------


In [17]:
urgent_tests = [
    "This is urgent",
    "This is URGENT",
    "This is Urgent",
    "This is important"
]

for text in urgent_tests:
    print(text, "->", check_urgent(text))

This is urgent -> True
This is URGENT -> True
This is Urgent -> True
This is important -> False


Task 4 — Limitation
These regular expressions may not recognize every possible email address, URL, or phone number format. The phone pattern only matches numbers starting with 010 followed by exactly eight digits, so numbers from other prefixes will not be detected. Also, the number extraction pattern may capture digits from order IDs and URLs, not just standalone numbers.